# Descarga de velas (klines) de Binance

Descarga velas spot desde `data.binance.vision` (público, sin API key), verifica el SHA-256 de cada archivo y guarda un CSV con el rango pedido.

- Meses cerrados: archivo mensual. Meses aún no publicados como mensual: archivos diarios.
- Desde 2025 los timestamps de spot vienen en microsegundos; se detecta por magnitud.
- Tiempos en UTC.

## 1. Imports

In [1]:
import hashlib
import io
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

## 1.1 [CONFIG]

In [2]:
RUTA         = Path(r"C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\reales\raw\cripto_binance")
PAR          = "BTCUSDT"
INTERVALO    = "5m"            # 1m, 3m, 5m, 15m, 30m, 1h, 2h, 4h, 6h, 8h, 12h, 1d
FECHA_INICIO = "2020-01-01"    # inclusive, UTC
FECHA_FIN    = "2026-08-31"    # inclusive, UTC

## 2. Funciones

In [3]:
BASE = "https://data.binance.vision/data/spot"
COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
        "quote_volume", "n_trades", "taker_buy_base", "taker_buy_quote", "ignore"]


def _bajar(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return r.read()


def obtener_zip(frecuencia, etiqueta, cache):
    """Zip verificado (bytes) o None si no existe. frecuencia: 'monthly' | 'daily'."""
    nombre = f"{PAR}-{INTERVALO}-{etiqueta}.zip"
    ruta = cache / frecuencia / nombre
    if ruta.exists():
        return ruta.read_bytes()
    url = f"{BASE}/{frecuencia}/klines/{PAR}/{INTERVALO}/{nombre}"
    try:
        datos = _bajar(url)
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return None
        raise
    sha = _bajar(url + ".CHECKSUM").decode().split()[0]
    if hashlib.sha256(datos).hexdigest() != sha:
        raise ValueError(f"checksum no coincide: {nombre}")
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_bytes(datos)
    return datos


def leer_zip(datos):
    with zipfile.ZipFile(io.BytesIO(datos)) as z:
        df = pd.read_csv(z.open(z.namelist()[0]), header=None, names=COLS)
    if not str(df.iloc[0, 0]).isdigit():   # algunos archivos traen encabezado
        df = df.iloc[1:].astype({c: float for c in COLS})
    for c in ("open_time", "close_time"):
        unidad = "us" if df[c].iloc[0] > 1e14 else "ms"
        df[c] = pd.to_datetime(df[c].astype("int64"), unit=unidad, utc=True)
    return df.drop(columns="ignore")

## 3. Descarga

In [4]:
inicio = pd.Timestamp(FECHA_INICIO, tz="UTC")
fin    = pd.Timestamp(FECHA_FIN, tz="UTC")
assert inicio <= fin, "FECHA_INICIO debe ser <= FECHA_FIN"

cache = RUTA / PAR / INTERVALO / "zip"
partes, faltantes = [], []

for mes in pd.period_range(inicio.tz_localize(None), fin.tz_localize(None), freq="M"):
    datos = obtener_zip("monthly", mes.strftime("%Y-%m"), cache)
    if datos is not None:
        partes.append(leer_zip(datos))
        print("mensual", mes)
        continue
    # mes aún no publicado como mensual: días sueltos
    dias = pd.date_range(max(mes.start_time, inicio.tz_localize(None)),
                         min(mes.end_time, fin.tz_localize(None)), freq="D")
    for dia in dias:
        datos = obtener_zip("daily", dia.strftime("%Y-%m-%d"), cache)
        if datos is None:
            faltantes.append(dia.date())
        else:
            partes.append(leer_zip(datos))
    print("diario ", mes, f"({len(dias)} días)")

assert partes, "no se descargó nada: revisar PAR, INTERVALO y fechas"
velas = (pd.concat(partes)
         .drop_duplicates("open_time")
         .sort_values("open_time")
         .loc[lambda d: (d["open_time"] >= inicio) & (d["open_time"] < fin + pd.Timedelta(days=1))]
         .reset_index(drop=True))
print("días sin archivo:", faltantes if faltantes else "ninguno")

mensual 2020-01
mensual 2020-02
mensual 2020-03
mensual 2020-04
mensual 2020-05
mensual 2020-06
mensual 2020-07
mensual 2020-08
mensual 2020-09
mensual 2020-10
mensual 2020-11
mensual 2020-12
mensual 2021-01
mensual 2021-02
mensual 2021-03
mensual 2021-04
mensual 2021-05
mensual 2021-06
mensual 2021-07
mensual 2021-08
mensual 2021-09
mensual 2021-10
mensual 2021-11
mensual 2021-12
mensual 2022-01
mensual 2022-02
mensual 2022-03
mensual 2022-04
mensual 2022-05
mensual 2022-06
mensual 2022-07
mensual 2022-08
mensual 2022-09
mensual 2022-10
mensual 2022-11
mensual 2022-12
mensual 2023-01
mensual 2023-02
mensual 2023-03
mensual 2023-04
mensual 2023-05
mensual 2023-06
mensual 2023-07
mensual 2023-08
mensual 2023-09
mensual 2023-10
mensual 2023-11
mensual 2023-12
mensual 2024-01
mensual 2024-02
mensual 2024-03
mensual 2024-04
mensual 2024-05
mensual 2024-06
mensual 2024-07
mensual 2024-08
mensual 2024-09
mensual 2024-10
mensual 2024-11
mensual 2024-12
mensual 2025-01
mensual 2025-02
mensual 

## 4. Guardado y resumen

In [5]:
archivo = RUTA / f"{PAR}_{INTERVALO}_{inicio:%Y%m%d}_{fin:%Y%m%d}.csv"
velas.to_csv(archivo, index=False)

por_dia = velas.groupby(velas["open_time"].dt.floor("D")).size()
esperado = por_dia.mode().iloc[0]
print(archivo.resolve())
print(f"velas: {len(velas):,}   días: {len(por_dia)}   velas/día esperadas: {esperado}")
print(f"días incompletos: {(por_dia < esperado).sum()}")
velas.head()

C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\reales\raw\cripto_binance\BTCUSDT_5m_20200101_20260831.csv
velas: 700,818   días: 2435   velas/día esperadas: 288
días incompletos: 15


,open_time,open,high,low,close,volume,close_time,quote_volume,n_trades,taker_buy_base,taker_buy_quote
0,2020-01-01 00:00:00+00:00,7195.24,7196.25,7178.64,7179.78,95.509133,2020-01-01 00:04:59.999000+00:00,686317.136252,1127,32.773245,235537.295045
1,2020-01-01 00:05:00+00:00,7179.76,7191.77,7178.20,7191.07,59.365225,2020-01-01 00:09:59.999000+00:00,426481.260364,631,24.766513,177935.618201
2,2020-01-01 00:10:00+00:00,7193.15,7193.53,7180.24,7180.97,48.068510,2020-01-01 00:14:59.999000+00:00,345446.503019,694,19.422283,139596.621683
3,2020-01-01 00:15:00+00:00,7180.97,7186.40,7177.35,7178.29,32.192929,2020-01-01 00:19:59.999000+00:00,231162.555424,576,12.963258,93091.433276
4,2020-01-01 00:20:00+00:00,7177.71,7182.46,7175.47,7176.96,49.027397,2020-01-01 00:24:59.999000+00:00,351927.893881,710,22.819744,163817.881155
